# console

In [ ]:
///- --test static

open testing

## fsharp

### console_color

In [ ]:
nominal console_color = $'System.ConsoleColor'

### reset_color

In [ ]:
inl reset_color () : () =
    run_target function
        | Fsharp => fun () => $'System.Console.ResetColor ()'
        | _ => fun () => ()

### set_foreground_color

In [ ]:
inl set_foreground_color (color : console_color) : () =
    run_target function
        | Fsharp => fun () => $'System.Console.ForegroundColor <- !color '
        | _ => fun () => ()

## console

### write_line

In [ ]:
inl write_line_c forall t. (obj : t) : () =
    real
        typecase t with
        | string => $'printf("%s\\n", (!obj)->ptr)' : ()
        | i32 => $'printf("%d\\n", !obj)' : ()
        | i64 => $'printf("%lld\\n", (long long)(!obj))' : ()
        | u32 => $'printf("%u\\n", !obj)' : ()
        | u64 => $'printf("%llu\\n", (unsigned long long)(!obj))' : ()
        | _ => $'backend_switch_has_no_C_arm_in_lib_spiral' : ()

inl write_line_gleam forall t. (obj : t) : () =
    global "import gleam/io"
    if (real typecase t with string => true | _ => false) then $'io.println(!obj)' : ()
    elif (real typecase t with () => true | _ => false) then $'io.println("")' : ()
    else
        global "import gleam/int"
        $'io.println(int.to_string(!obj))' : ()

inl write_wasm forall t. (obj : t) : () =
    real
        typecase t with
        | string => $'(call $print_str !obj)' : ()
        | i32 => $'(call $print_i64 (i64.extend_i32_s !obj))' : ()
        | i64 => $'(call $print_i64 !obj)' : ()
        | u32 => $'(call $print_u64 (i64.extend_i32_u !obj))' : ()
        | u64 => $'(call $print_u64 !obj)' : ()
        | () => $'' : ()
        | _ => $'backend_switch_has_no_Wasm_arm_in_lib_spiral' : ()

inl write_line_bend forall t. (x : t) : () =
    real
        typecase t with
        | string => !!!!Printf("%s\n", x)
        | i32 => !!!!Printf("%d\n", x)
        | _ => $'backend_switch_has_no_Bend_arm_in_lib_spiral' : ()

inl write_line_lua forall t. (obj : t) : () =
    real
        typecase t with
        | i64 => $'print(spiral_w64.tostring(!obj, true))' : ()
        | u64 => $'print(spiral_w64.tostring(!obj, false))' : ()
        | _ => $'print(!obj)' : ()

inl write_line forall t. (obj : t) : () =
    backend_switch {
        Gleam = fun () => write_line_gleam obj
        Lua = fun () => write_line_lua obj
        Fsharp = fun () =>
            fun () => obj |> $'System.Console.WriteLine'
            |> exec_unit
        Python = fun () => $'print(!obj)' : ()
        Rust = fun () => $'println\!("{}", !obj)' : ()
        TypeScript = fun () => $'process.stdout.write(String(!obj) + "\\n")' : ()
        Zig = fun () => $'{ spiralPrintAny(!obj); spiralPrint(&[_]u8{10}); }' : ()
        Lean = fun () => $'IO.println (toString !obj)' : ()
        C = fun () => write_line_c obj
        Delphi = fun () => $'Writeln(!obj)' : ()
        Kotlin = fun () => $'run { System.out.write((!obj.toString() + "\\n").toByteArray(Charsets.UTF_8)); System.out.flush() }' : ()
        Swift = fun () => $'print(!obj)' : ()
        Bend = fun () => write_line_bend obj
        Wasm = fun () =>
            write_wasm obj
            $'(call $print_newline)' : ()
    }

### write

In [ ]:
inl write_c forall t. (x : t) : () =
    real
        typecase t with
        | string => $'printf("%s", (!x)->ptr)' : ()
        | i32 => $'printf("%d", !x)' : ()
        | i64 => $'printf("%lld", (long long)(!x))' : ()
        | u32 => $'printf("%u", !x)' : ()
        | u64 => $'printf("%llu", (unsigned long long)(!x))' : ()
        | _ => $'backend_switch_has_no_C_arm_in_lib_spiral' : ()

inl write_gleam forall t. (x : t) : () =
    global "import gleam/io"
    if (real typecase t with string => true | _ => false) then $'io.print(!x)' : ()
    else
        global "import gleam/int"
        $'io.print(int.to_string(!x))' : ()

inl write_lua forall t. (x : t) : () =
    real
        typecase t with
        | i64 => $'io.write(spiral_w64.tostring(!x, true))' : ()
        | u64 => $'io.write(spiral_w64.tostring(!x, false))' : ()
        | _ => $'io.write(!x)' : ()

inl write forall t. (x : t) : () =
    backend_switch {
        Gleam = fun () => write_gleam x
        Lua = fun () => write_lua x
        Fsharp = fun () => $'!x |> System.Console.Write' : ()
        Python = fun () => $'print(!x, end="")' : ()
        Rust = fun () => $'{ use std::io::Write; print\!("{}", !x); std::io::stdout().flush().unwrap(); }' : ()
        TypeScript = fun () => $'process.stdout.write(String(!x))' : ()
        Zig = fun () => $'spiralPrintAny(!x)' : ()
        Lean = fun () => $'IO.print (toString !x)' : ()
        C = fun () => write_c x
        Delphi = fun () => $'Write(!x)' : ()
        Kotlin = fun () => $'run { System.out.write(!x.toString().toByteArray(Charsets.UTF_8)); System.out.flush() }' : ()
        Swift = fun () => $'print(!x, terminator: "")' : ()
        Wasm = fun () => write_wasm x
    }

### write_ln

In [ ]:
inl write_ln l : () =
    write l
    backend_switch {
        Gleam = fun () => write_line () : ()
        Lua = fun () => write_line () : ()
        Fsharp = fun () => write_line () : ()
        Python = fun () => $"print()" : ()
        Cuda = fun () => $'printf("\\n")' : ()
        Rust = fun () => $'println\!()' : ()
        Zig = fun () => $'spiralPrint(&[_]u8{10})' : ()
        Lean = fun () => $'IO.println ""' : ()
    }

### color_reset

In [ ]:
inl color_reset () =
    run_target function
        | Zig => fun () =>
            $'"\\u{001b}[0m"' : string
        | Gleam _ => fun () => $'"\\u{001b}[0m"'
        | Lua _ => fun () => $'""'
        | Rust => fun () => $'Rc::<str>::from("\\u{001b}[0m")' : string
        | _ => fun () => $'"\\u001b[0m"'

### color_bright_black

In [ ]:
inl color_bright_black () =
    run_target function
        | Zig => fun () =>
            $'"\\u{001b}[90m"' : string
        | Gleam _ => fun () => $'"\\u{001b}[90m"'
        | Lua _ => fun () => $'""'
        | Rust => fun () => $'Rc::<str>::from("\\u{001b}[90m")' : string
        | _ => fun () => $'"\\u001b[90m"'

### color_bright_red

In [ ]:
inl color_bright_red () =
    run_target function
        | Zig => fun () =>
            $'"\\u{001b}[91m"' : string
        | Gleam _ => fun () => $'"\\u{001b}[91m"'
        | Lua _ => fun () => $'""'
        | Rust => fun () => $'Rc::<str>::from("\\u{001b}[91m")' : string
        | _ => fun () => $'"\\u001b[91m"'

### color_bright_green

In [ ]:
inl color_bright_green () =
    run_target function
        | Zig => fun () =>
            $'"\\u{001b}[92m"' : string
        | Gleam _ => fun () => $'"\\u{001b}[92m"'
        | Lua _ => fun () => $'""'
        | Rust => fun () => $'Rc::<str>::from("\\u{001b}[92m")' : string
        | _ => fun () => $'"\\u001b[92m"'

### color_yellow

In [ ]:
inl color_yellow () =
    run_target function
        | Zig => fun () =>
            $'"\\u{001b}[93m"' : string
        | Gleam _ => fun () => $'"\\u{001b}[93m"'
        | Lua _ => fun () => $'""'
        | Rust => fun () => $'Rc::<str>::from("\\u{001b}[93m")' : string
        | _ => fun () => $'"\\u001b[93m"'

### color_bright_blue

In [ ]:
inl color_bright_blue () =
    run_target function
        | Zig => fun () =>
            $'"\\u{001b}[94m"' : string
        | Gleam _ => fun () => $'"\\u{001b}[94m"'
        | Lua _ => fun () => $'""'
        | Rust => fun () => $'Rc::<str>::from("\\u{001b}[94m")' : string
        | _ => fun () => $'"\\u001b[94m"'

### color_bright_magenta

In [ ]:
inl color_bright_magenta () =
    run_target function
        | Zig => fun () =>
            $'"\\u{001b}[95m"' : string
        | Gleam _ => fun () => $'"\\u{001b}[95m"'
        | Lua _ => fun () => $'""'
        | Rust => fun () => $'Rc::<str>::from("\\u{001b}[95m")' : string
        | _ => fun () => $'"\\u001b[95m"'

### color_bright_cyan

In [ ]:
inl color_bright_cyan () =
    run_target function
        | Zig => fun () =>
            $'"\\u{001b}[96m"' : string
        | Gleam _ => fun () => $'"\\u{001b}[96m"'
        | Lua _ => fun () => $'""'
        | Rust => fun () => $'Rc::<str>::from("\\u{001b}[96m")' : string
        | _ => fun () => $'"\\u001b[96m"'

### color_bright_white

In [1]:
inl color_bright_white () =
    run_target function
        | Zig => fun () =>
            $'"\\u{001b}[97m"' : string
        | Gleam _ => fun () => $'"\\u{001b}[97m"'
        | Lua _ => fun () => $'""'
        | Rust => fun () => $'Rc::<str>::from("\\u{001b}[97m")' : string
        | _ => fun () => $'"\\u001b[97m"'